In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import pyvista as pv
from mat4py import loadmat
import re
from itertools import chain
import seaborn as sns
import ollama
from tqdm import tqdm
from collections import defaultdict
import kagglehub

def extract_ingredients(s):
    return [ingredient.lower() for ingredient in re.findall(r'"(.*?)"', s)]


### Load the entire dataset and look at a few statistics

In [15]:
path1 = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews") + '/recipes.csv'
df1 = pd.read_csv(path1)

path2 = kagglehub.dataset_download("realalexanderwei/food-com-recipes-with-ingredients-and-tags")  + '/recipes_ingredients.csv'
df2 = pd.read_csv(path2)

# Ensure matching data types
df1['RecipeId'] = pd.to_numeric(df1['RecipeId'], errors='coerce')
df2['id'] = pd.to_numeric(df2['id'], errors='coerce')

# Drop duplicates: keep only the first ingredient entry per recipe
df2_unique = df2[['id', 'ingredients_raw', 'ingredients']].drop_duplicates(subset='id')

# Merge
data = df1.merge(df2_unique, left_on='RecipeId', right_on='id', how='left')
data.drop(columns='id', inplace=True)

In [16]:
# data['ParsedIngredients'] = data['RecipeIngredientParts'].apply(extract_ingredients)
# ingredients = [eval(item) for item in data['ingredients']]
# data['NumIngredients'] = data['ParsedIngredients'].apply(len)
categories = data['RecipeCategory'].unique().tolist()
authors = data['AuthorId'].unique().tolist()

print('Statistics:')
print('Recipes: ', len(data))
print('Authors: ', len(authors))
print('Food categories: ', len(categories))
# print('Ingredients: ', len(ingredients))
# print('Ingredients per food:', 
#       ' mean:', np.mean(data['NumIngredients']), 
#       ' median: ', np.median(data['NumIngredients']), 
#       ' min: ', np.min(data['NumIngredients']), 
#       ' max: ', np.max(data['NumIngredients']))

Statistics:
Recipes:  522517
Authors:  57178
Food categories:  312


### Extract and process a few distinct food categories

In [3]:
isburger_llm = pd.read_csv('data/isburger_llm.csv')
iscurry_llm = pd.read_csv('data/iscurry_llm.csv')
isomelette_llm = pd.read_csv('data/isomelette_llm.csv')
issoup_llm = pd.read_csv('data/issoup_llm.csv')
data['isburger_llm'] = isburger_llm['isburger_llm']
data['iscurry_llm'] = iscurry_llm['iscurry_llm']
data['isomelette_llm'] = isomelette_llm['isomelette_llm']
data['issoup_llm'] = issoup_llm['issoup_llm']

data['isburger_search'] = 0
data['iscurry_search'] = 0
data['isomelette_search'] = 0
data['issoup_search'] = 0
for idx, row in data.iterrows():
    name = row['Name'].lower() if isinstance(row['Name'], str) else ''
    if 'burger' in name:
        data.at[idx, 'isburger_search'] = 1
    if 'curry' in name:
        data.at[idx, 'iscurry_search'] = 1
    if 'omelet' in name:
        data.at[idx, 'isomelette_search'] = 1
    if 'soup' in name:
        data.at[idx, 'issoup_search'] = 1

In [4]:
def convert_str_to_int(val):
    if isinstance(val, str):
        try:
            # Try converting the whole string
            return int(val)
        except ValueError:
            try:
                # Try converting the first two characters
                return int(val[:2])
            except (ValueError, TypeError):
                # Third attempt: look for '1' or '0' in the string
                for char in val:
                    if char in ('0', '1'):
                        return int(char)
                return 0
    return val

# Rows that couldn't be converted to int
data["isburger_llm"] = data["isburger_llm"].apply(convert_str_to_int)
data["iscurry_llm"] = data["iscurry_llm"].apply(convert_str_to_int)
data["isomelette_llm"] = data["isomelette_llm"].apply(convert_str_to_int)
data["issoup_llm"] = data["issoup_llm"].apply(convert_str_to_int)

data['isburger'] = ((data['isburger_llm'] == 1) & (data['isburger_search'] == 1)).astype(int)
burgers = data[data['isburger'] == 1].reset_index(drop=True)
burgers = burgers[burgers['ingredients_raw'].notna()].reset_index(drop=True)

data['iscurry'] = ((data['iscurry_llm'] == 1) & (data['iscurry_search'] == 1)).astype(int)
curries = data[data['iscurry'] == 1].reset_index(drop=True)
curries = curries[curries['ingredients_raw'].notna()].reset_index(drop=True)

data['isomelette'] = ((data['isomelette_llm'] == 1) & (data['isomelette_search'] == 1)).astype(int)
omelettes = data[data['isomelette'] == 1].reset_index(drop=True)
omelettes = omelettes[omelettes['ingredients_raw'].notna()].reset_index(drop=True)

data['issoup'] = ((data['issoup_llm'] == 1) & (data['issoup_search'] == 1)).astype(int)
soups = data[data['issoup'] == 1].reset_index(drop=True)
soups = soups[soups['ingredients_raw'].notna()].reset_index(drop=True)

columns_to_drop = ['RecipeId', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Images',
                   'RecipeCategory', 'Keywords', 'isburger_llm', 'iscurry_llm', 'isomelette_llm', 'issoup_llm', 'isburger_search',
                   'iscurry_search', 'isomelette_search', 'issoup_search', 'isburger']  # Add more as needed
burgers = burgers.drop(columns=columns_to_drop)
curries = curries.drop(columns=columns_to_drop)
omelettes = omelettes.drop(columns=columns_to_drop)
soups = soups.drop(columns=columns_to_drop)

In [5]:
bg_ingredients = set(chain.from_iterable(burgers['RecipeIngredientParts'].apply(extract_ingredients)))
print('Burger recipes: ', len(burgers), ', ingredients:', len(bg_ingredients), ', Recipe to ingredient ratio: ', len(burgers)/len(bg_ingredients))

cr_ingredients = set(chain.from_iterable(curries['RecipeIngredientParts'].apply(extract_ingredients)))
print('Curry recipes: ', len(curries), ', ingredients:', len(cr_ingredients), ', Recipe to ingredient ratio: ', len(curries)/len(cr_ingredients))

om_ingredients = set(chain.from_iterable(omelettes['RecipeIngredientParts'].apply(extract_ingredients)))
print('Omelette recipes: ', len(omelettes), ', ingredients:', len(om_ingredients), ', Recipe to ingredient ratio: ', len(omelettes)/len(om_ingredients))

sp_ingredients = set(chain.from_iterable(soups['RecipeIngredientParts'].apply(extract_ingredients)))
print('Soup recipes: ', len(soups), ', ingredients:', len(sp_ingredients), ', Recipe to ingredient ratio: ', len(soups)/len(sp_ingredients))

burgers.to_csv('data/burgers.csv', index=False)
curries.to_csv('data/curries.csv', index=False)
omelettes.to_csv('data/omelettes.csv', index=False)
soups.to_csv('data/soups.csv', index=False)

Burger recipes:  3882 , ingredients: 1629 , Recipe to ingredient ratio:  2.3830570902394106
Curry recipes:  3288 , ingredients: 1603 , Recipe to ingredient ratio:  2.0511540860885837
Omelette recipes:  1011 , ingredients: 796 , Recipe to ingredient ratio:  1.270100502512563
Soup recipes:  20595 , ingredients: 3019 , Recipe to ingredient ratio:  6.82179529645578
